In [1]:
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Get the project root folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Start a local Spark session
spark = (
    SparkSession.builder
    .appName("MLlib-Model-Training")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "10")
    .getOrCreate()
)

print("SparkSession started.")

SparkSession started.


In [2]:
from pyspark.sql import functions as F

def train_saturation_model(aggregated_traffic_df):

    # Set a maximum capacity for each infrastructure type
    df = aggregated_traffic_df.withColumn(
        "max_capacity",
        F.when(F.col("infrastructure_type") == "bike_path", 3000.0)
         .when(F.col("infrastructure_type") == "pedestrian_zone", 600.0)
         .otherwise(1000.0)
    )

    # Calculate the saturation index
    df = df.withColumn(
        "saturation_index",
        F.col("count").cast("double") / F.col("max_capacity")
    )

    # Convert text values into numeric values
    indexer = StringIndexer(
        inputCol="infrastructure_type",
        outputCol="infra_type_encoded"
    )
    df = indexer.fit(df).transform(df)

    # Create the feature vector
    assembler = VectorAssembler(
        inputCols=["infra_type_encoded", "count"],
        outputCol="features"
    )

    data = (
        assembler.transform(df)
        .select("features", "saturation_index")
        .withColumnRenamed("saturation_index", "label")
    )

    # Split the data into training and test sets
    train_data, test_data = data.randomSplit([0.7, 0.3], seed=123)

    # Train the linear regression model
    model = LinearRegression(
        featuresCol="features",
        labelCol="label",
        regParam=0.01
    ).fit(train_data)

    return model, test_data

In [3]:
def evaluate_model(model, test_data):

    # Make predictions on the test data
    predictions = model.transform(test_data)

    # Evaluate the model
    rmse = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="rmse"
    ).evaluate(predictions)

    r2 = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="r2"
    ).evaluate(predictions)

    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}")

    return predictions

In [4]:
def save_and_load_model(model, path="models/saturation_linear_regression"):

    # Save the trained model
    full_path = PROJECT_ROOT / path
    model.write().overwrite().save(str(full_path))

In [5]:
# Load the traffic data
simulation_path = PROJECT_ROOT / "data" / "geospatial_output"

traffic_spatial_df = spark.read.csv(
    str(simulation_path / "infrastructure_activity_counts.csv"),
    header=True,
    inferSchema=True
)

# Train and evaluate the model
model, test_data = train_saturation_model(traffic_spatial_df)
predictions = evaluate_model(model, test_data)

# Save the trained model
save_and_load_model(model)

# Show some predictions
predictions.select(
    "features",
    "label",
    "prediction"
).show(5, truncate=False)

RMSE: 0.0472
R²: 0.8953
+-----------+------------------+------------------+
|features   |label             |prediction        |
+-----------+------------------+------------------+
|[0.0,461.0]|0.7683333333333333|0.7398064287417601|
|[0.0,500.0]|0.8333333333333334|0.7521640377186172|
|[0.0,500.0]|0.8333333333333334|0.7521640377186172|
|[1.0,500.0]|0.5               |0.4906295165043656|
|[1.0,500.0]|0.5               |0.4906295165043656|
+-----------+------------------+------------------+
only showing top 5 rows

